# 02 — Evaluate YOLO11 and build the object-crop dataset

This notebook validates the best segmentation checkpoint on a reproducible
review sample and then runs large-scale inference over the remaining VisA
images. Every expected-class detection is converted into a masked object crop
for downstream normal/defective classification.

## Pipeline position

`YOLO11 checkpoint → visual sample review → full-dataset segmentation → object crops`

## Label assignment

- A crop is **defective** when its predicted object mask overlaps the VisA
  anomaly mask by at least `MIN_INTERSECTION_PIXELS`.
- Otherwise it is **normal**.
- Only detections whose predicted product class matches the source dataset
  class are accepted.

## Main outputs

| Artifact | Purpose |
|---|---|
| `sample_results.csv` | Results from the reproducible review sample |
| `remaining_results.csv` | Checkpointed results for all unsampled images |
| `full_results.csv` | Combined object-level metadata |
| `binary_dataset/normal/` | Masked normal object crops |
| `binary_dataset/defective/` | Masked defective object crops |
| `comparisons/` | Original, ground-truth, and prediction panels |

Run the sample stage first and inspect its visual panels before starting the
full pass.


## 1. Imports, inference settings, paths, and model loading

This cell centralizes all reproducibility and performance settings. Set
`MODEL_PATH` to an explicit checkpoint when you do not want automatic selection
of the newest `best.pt`.

`RETINA_MASKS=True` preserves masks at source-image resolution, which keeps
pixel-overlap labeling consistent. Existing sample results can be reused with
`REUSE_SAMPLE_RESULTS=True`.


In [1]:
# Configure deterministic YOLO inference and locate project artifacts
from pathlib import Path
import gc
import math
import random
import warnings

import cv2
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from tqdm.auto import tqdm
from ultralytics import YOLO

# Reproducibility, inference quality, and output controls.
SEED = 42
SAMPLE_PER_CONDITION = 20
IMGSZ = 640
CONFIDENCE = 0.25
IOU = 0.70
MAX_DETECTIONS = 50
MIN_INTERSECTION_PIXELS = 1
SAVE_COMPARISONS = True
MODEL_PATH = None  # Prefer an explicit path for portable reruns.

CUDA = torch.cuda.is_available()
DEVICE = 0 if CUDA else "cpu"
BATCH_SIZE = 8 if CUDA else 4
PREDICTION_CHUNK_SIZE = 64
RETINA_MASKS = True  # Keep sample and remaining overlap calculations pixel-consistent.
REUSE_SAMPLE_RESULTS = True
USE_HALF = CUDA
if CUDA:
    torch.backends.cudnn.benchmark = True

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
RESULT_COLUMNS = [
    "source_image", "source_class", "condition", "status", "object_id",
    "predicted_class", "confidence", "assigned_label", "defect_pixels", "saved_path",
]

# Resolve the historical experiment root from common launch locations.
def find_ridac_dir():
    here = Path.cwd().resolve()
    candidates = [here, here / "misc" / "ridac"]
    candidates.extend(parent / "misc" / "ridac" for parent in here.parents)
    for candidate in dict.fromkeys(candidates):
        if (candidate / "extracted_dataset").is_dir():
            return candidate
    raise FileNotFoundError("Could not find misc/ridac/extracted_dataset from the current directory.")

def find_repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists() or (candidate / "deep_trial").is_dir():
            return candidate
    return start

# Use an override when supplied; otherwise select the newest trained run.
def latest_checkpoint(repo_root, override=None):
    if override is not None:
        checkpoint = Path(override).expanduser().resolve()
        if not checkpoint.is_file():
            raise FileNotFoundError(f"MODEL_PATH does not exist: {checkpoint}")
        return checkpoint
    run_roots = [
        repo_root / "deep_trial" / "Red-Light-Violation-Detection" / "runs" / "segment",
        repo_root / "runs" / "segment",
    ]
    checkpoints = [
        checkpoint
        for root in run_roots if root.is_dir()
        for checkpoint in root.glob("train*/weights/best.pt")
    ]
    if not checkpoints:
        searched = "\n".join(map(str, run_roots))
        raise FileNotFoundError(f"No train*/weights/best.pt checkpoint found under:\n{searched}")
    return max(checkpoints, key=lambda checkpoint: checkpoint.stat().st_mtime)

RIDAC_DIR = find_ridac_dir()
REPO_ROOT = find_repo_root(RIDAC_DIR)
DATA_ROOT = RIDAC_DIR / "extracted_dataset"
BEST_MODEL = latest_checkpoint(REPO_ROOT, MODEL_PATH)
OUTPUT_ROOT = RIDAC_DIR / "object_defect_dataset"
BINARY_ROOT = OUTPUT_ROOT / "binary_dataset"
COMPARISON_ROOT = OUTPUT_ROOT / "comparisons"
RESULTS_SAMPLE = OUTPUT_ROOT / "sample_results.csv"
RESULTS_REMAINING = OUTPUT_ROOT / "remaining_results.csv"
RESULTS_FULL = OUTPUT_ROOT / "full_results.csv"
for folder in (BINARY_ROOT / "normal", BINARY_ROOT / "defective", COMPARISON_ROOT):
    folder.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
model = YOLO(str(BEST_MODEL))
print(f"Dataset: {DATA_ROOT}\nModel: {BEST_MODEL}\nDevice: {DEVICE}; batch={BATCH_SIZE}; half={USE_HALF}")


Dataset: /home/student/Deep-Mtech/misc/ridac/extracted_dataset
Model: /home/student/Deep-Mtech/deep_trial/Red-Light-Violation-Detection/runs/segment/train-7/weights/best.pt
Device: 0; batch=8; half=True


## 2. Build and validate the source-image manifest

The manifest contains one row per source image with its product class,
condition, image path, and optional anomaly-mask path. The checkpoint class
mapping is canonicalized so the historical model name `wheel` maps to the VisA
class `fryum`.

The displayed class/condition table is a dataset sanity check. Missing model
classes stop execution before expensive inference begins.


In [2]:
# Discover VisA classes and construct the complete image manifest
# Normalize class names and create one metadata row per source image.
# Resolve historical naming differences between checkpoints and data.
def canonical_class(name):
    normalized = str(name).strip().lower().replace("-", "_").replace(" ", "_")
    aliases = {"wheel": "fryum"}
    return aliases.get(normalized, normalized)

def image_files(folder):
    return sorted(
        path for path in folder.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    )

def get_classes():
    return sorted(
        path.name for path in DATA_ROOT.iterdir()
        if path.is_dir() and (path / "Data" / "Images" / "Normal").is_dir()
    )

CLASS_NAMES = get_classes()
if not CLASS_NAMES:
    raise RuntimeError(f"No dataset classes found under {DATA_ROOT}")

raw_model_names = model.names
if isinstance(raw_model_names, dict):
    MODEL_NAMES = {int(class_id): str(name) for class_id, name in raw_model_names.items()}
else:
    MODEL_NAMES = dict(enumerate(map(str, raw_model_names)))
MODEL_CANONICAL = {class_id: canonical_class(name) for class_id, name in MODEL_NAMES.items()}
dataset_canonical = {canonical_class(name) for name in CLASS_NAMES}
missing_classes = dataset_canonical - set(MODEL_CANONICAL.values())
if missing_classes:
    raise ValueError(
        f"Checkpoint has no mapping for {sorted(missing_classes)}. Model classes: {MODEL_NAMES}"
    )

def find_mask(mask_dir, stem):
    direct = mask_dir / f"{stem}.png"
    if direct.is_file():
        return direct
    matches = [
        path for path in mask_dir.glob(f"{stem}.*")
        if path.suffix.lower() in IMAGE_SUFFIXES
    ]
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one mask for {stem!r} in {mask_dir}; found {len(matches)}"
        )
    return matches[0]

# Pair anomaly images with masks; normal images intentionally have none.
def build_manifest():
    records = []
    for class_name in CLASS_NAMES:
        data_dir = DATA_ROOT / class_name / "Data"
        for condition in ("Normal", "Anomaly"):
            images = image_files(data_dir / "Images" / condition)
            mask_dir = data_dir / "Masks" / "Anomaly"
            for image_path in images:
                mask_path = find_mask(mask_dir, image_path.stem) if condition == "Anomaly" else None
                records.append({
                    "class": class_name,
                    "condition": condition,
                    "image": str(image_path.resolve()),
                    "mask": str(mask_path.resolve()) if mask_path else "",
                })
    manifest = pd.DataFrame.from_records(records, columns=["class", "condition", "image", "mask"])
    if manifest.empty:
        raise RuntimeError("The dataset manifest is empty.")
    return manifest

FULL_MANIFEST = build_manifest()
print("Model class mapping:", MODEL_CANONICAL)
display(FULL_MANIFEST.groupby(["class", "condition"]).size().unstack(fill_value=0))


Model class mapping: {0: 'candle', 1: 'capsules', 2: 'cashew', 3: 'chewinggum', 4: 'fryum', 5: 'macaroni1', 6: 'macaroni2', 7: 'pcb1', 8: 'pcb2', 9: 'pcb3', 10: 'pcb4', 11: 'pipe_fryum'}


condition,Anomaly,Normal
class,,
candle,100,1000
capsules,100,602
cashew,100,500
chewinggum,100,503
fryum,100,500
macaroni1,100,1000
macaroni2,100,1000
pcb1,100,1004
pcb2,100,1001


## 3. Image, mask, crop, and visualization utilities

These helpers perform four jobs:

1. Load and resize ground-truth anomaly masks.
2. Convert Ultralytics tensors into NumPy masks, class IDs, confidences, and
   boxes.
3. Remove the background and crop each predicted object.
4. Produce side-by-side visual review panels.

Nearest-neighbor interpolation is used for masks so resizing does not create
fractional labels.


In [3]:
# Define mask conversion, crop extraction, and review-panel helpers
# Keep binary masks discrete and preserve only predicted foreground.
def load_mask(mask_path, image_shape):
    height, width = image_shape[:2]
    if not mask_path:
        return np.zeros((height, width), dtype=bool)
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise OSError(f"Could not read mask: {mask_path}")
    if mask.shape != (height, width):
        mask = cv2.resize(mask, (width, height), interpolation=cv2.INTER_NEAREST)
    return mask > 0

def result_arrays(result, image_shape):
    if result.masks is None or result.boxes is None or len(result.boxes) == 0:
        return None
    masks = result.masks.data.detach().cpu().numpy()
    class_ids = result.boxes.cls.detach().cpu().numpy().astype(np.int32, copy=False)
    confidences = result.boxes.conf.detach().cpu().numpy()
    boxes = result.boxes.xyxy.detach().cpu().numpy()
    height, width = image_shape[:2]
    if masks.shape[1:] != (height, width):
        masks = np.stack([
            cv2.resize(mask, (width, height), interpolation=cv2.INTER_NEAREST)
            for mask in masks
        ])
    return masks > 0.5, class_ids, confidences, boxes

# Clip the box to image bounds and black out all non-object pixels.
def masked_crop(image, mask, box):
    height, width = image.shape[:2]
    x1, y1, x2, y2 = np.rint(box).astype(int)
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(width, x2), min(height, y2)
    if x2 <= x1 or y2 <= y1:
        return None
    region = image[y1:y2, x1:x2]
    region_mask = mask[y1:y2, x1:x2].astype(np.uint8, copy=False) * 255
    if not np.any(region_mask):
        return None
    return cv2.bitwise_and(region, region, mask=region_mask)

def boundary_overlay(image, masks, color):
    canvas = image.copy()
    if masks is None:
        return canvas
    union = masks if masks.ndim == 2 else np.any(masks, axis=0)
    contours, _ = cv2.findContours(
        union.astype(np.uint8) * 255, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    cv2.drawContours(canvas, contours, -1, color, 2, lineType=cv2.LINE_AA)
    return canvas

def add_title(image, title):
    canvas = image.copy()
    cv2.rectangle(canvas, (0, 0), (canvas.shape[1], 34), (0, 0, 0), -1)
    cv2.putText(
        canvas, title, (10, 24), cv2.FONT_HERSHEY_SIMPLEX,
        0.65, (255, 255, 255), 2, cv2.LINE_AA,
    )
    return canvas

# Save original, ground-truth boundary, and prediction boundary panels.
def save_comparison(image, gt_mask, prediction_masks, destination):
    panels = [
        add_title(image, "Original"),
        add_title(boundary_overlay(image, gt_mask, (0, 255, 0)), "Ground truth"),
        add_title(
            boundary_overlay(image, prediction_masks, (0, 0, 255)),
            "Expected-class prediction",
        ),
    ]
    ok = cv2.imwrite(
        str(destination), np.concatenate(panels, axis=1),
        [cv2.IMWRITE_JPEG_QUALITY, 92],
    )
    if not ok:
        warnings.warn(f"Could not write comparison: {destination}")


## 4. Core inference, labeling, export, and resume logic

`process_dataset` streams YOLO predictions and emits one result row per accepted
object. It also records explicit failure statuses such as unreadable images,
missing expected-class detections, empty masks, and write failures.

`process_dataset_resumable` handles the large run in small chunks. After each
chunk it atomically replaces the CSV checkpoint, so an interrupted run resumes
without repeating every completed image.


In [4]:
# Define the object-level inference and checkpointed export pipeline
# Each output row is traceable to its source image and predicted object.
def result_record(image_path, class_name, condition, status, **values):
    record = {
        "source_image": image_path,
        "source_class": class_name,
        "condition": condition,
        "status": status,
        "object_id": pd.NA,
        "predicted_class": "",
        "confidence": np.nan,
        "assigned_label": "",
        "defect_pixels": 0,
        "saved_path": "",
    }
    record.update(values)
    return record

def process_dataset(manifest, save_csv, save_comparisons=False, report=True):
    manifest = manifest[["class", "condition", "image", "mask"]].reset_index(drop=True)
    if manifest.empty:
        empty = pd.DataFrame(columns=RESULT_COLUMNS)
        empty.to_csv(save_csv, index=False)
        return empty

    predictions = model.predict(
        source=manifest["image"].tolist(),
        imgsz=IMGSZ,
        conf=CONFIDENCE,
        iou=IOU,
        max_det=MAX_DETECTIONS,
        device=DEVICE,
        batch=BATCH_SIZE,
        half=USE_HALF,
        retina_masks=RETINA_MASKS,
        stream=True,
        verbose=False,
    )
    records = []
    rows = manifest.itertuples(index=False, name=None)
    iterator = zip(rows, predictions)
    for values, result in tqdm(iterator, total=len(manifest), desc="Segmenting", unit="image"):
        class_name, condition, image_path, mask_path = values
        image = cv2.imread(image_path, cv2.IMREAD_COLOR)
        if image is None:
            records.append(result_record(image_path, class_name, condition, "unreadable_image"))
            continue

        gt_mask = load_mask(mask_path, image.shape)
        arrays = result_arrays(result, image.shape)
        expected = canonical_class(class_name)
        if arrays is None:
            masks = np.empty((0, *image.shape[:2]), dtype=bool)
            class_ids = np.empty(0, dtype=np.int32)
            confidences = np.empty(0, dtype=np.float32)
            boxes = np.empty((0, 4), dtype=np.float32)
        else:
            masks, class_ids, confidences, boxes = arrays
        # Ignore detections for product classes that do not match the source.
        selected = [
            index for index, class_id in enumerate(class_ids)
            if MODEL_CANONICAL.get(int(class_id)) == expected
        ]

        if save_comparisons and SAVE_COMPARISONS:
            comparison_name = f"{class_name}_{condition}_{Path(image_path).stem}.jpg"
            prediction_masks = masks[selected] if selected else None
            save_comparison(
                image, gt_mask, prediction_masks, COMPARISON_ROOT / comparison_name
            )

        if not selected:
            records.append(
                result_record(image_path, class_name, condition, "no_expected_class_detection")
            )
            continue

        for object_id in selected:
            object_mask = masks[object_id]
            # Any configured overlap with the anomaly mask marks the crop defective.
            defect_pixels = int(np.count_nonzero(object_mask & gt_mask))
            label = "defective" if defect_pixels >= MIN_INTERSECTION_PIXELS else "normal"
            crop = masked_crop(image, object_mask, boxes[object_id])
            saved_path = ""
            status = "empty_mask"
            if crop is not None:
                filename = (
                    f"{class_name}_{condition}_{Path(image_path).stem}_obj_{object_id}.png"
                )
                destination = BINARY_ROOT / label / filename
                opposite_label = "normal" if label == "defective" else "defective"
                (BINARY_ROOT / opposite_label / filename).unlink(missing_ok=True)
                if cv2.imwrite(
                    str(destination), crop, [cv2.IMWRITE_PNG_COMPRESSION, 1]
                ):
                    status, saved_path = "ok", str(destination)
                else:
                    status = "write_failed"
            records.append(result_record(
                image_path,
                class_name,
                condition,
                status,
                object_id=object_id,
                predicted_class=MODEL_NAMES[int(class_ids[object_id])],
                confidence=float(confidences[object_id]),
                assigned_label=label,
                defect_pixels=defect_pixels,
                saved_path=saved_path,
            ))

    results = pd.DataFrame.from_records(records, columns=RESULT_COLUMNS)
    results.to_csv(save_csv, index=False)
    print(f"Saved {len(results):,} result rows to {save_csv}")
    if report:
        display(results["status"].value_counts(dropna=False).rename("rows"))
        display(
            results.loc[results["status"] == "ok", "assigned_label"]
            .value_counts()
            .rename("saved crops")
        )
    return results


def process_dataset_resumable(manifest, save_csv, chunk_size=PREDICTION_CHUNK_SIZE):
    """Process small chunks and atomically checkpoint after every chunk.

    Re-running this function skips every source image already present in save_csv.
    If execution stops within a chunk, only that small chunk is repeated.
    """
    save_csv = Path(save_csv)
    manifest = manifest[["class", "condition", "image", "mask"]].reset_index(drop=True)
    if save_csv.is_file():
        try:
            accumulated = pd.read_csv(save_csv)
        except pd.errors.EmptyDataError:
            accumulated = pd.DataFrame(columns=RESULT_COLUMNS)
    else:
        accumulated = pd.DataFrame(columns=RESULT_COLUMNS)

    completed_images = set(accumulated.get("source_image", pd.Series(dtype=str)).dropna())
    pending = manifest.loc[~manifest["image"].isin(completed_images)].reset_index(drop=True)
    print(
        f"Resume state: {len(completed_images):,} images completed; "
        f"{len(pending):,} images remaining."
    )
    if pending.empty:
        return accumulated

    chunk_csv = save_csv.with_suffix(".current_chunk.csv")
    chunk_starts = range(0, len(pending), chunk_size)
    for start in tqdm(
        chunk_starts,
        total=math.ceil(len(pending) / chunk_size),
        desc="Checkpointed chunks",
        unit="chunk",
    ):
        stop = min(start + chunk_size, len(pending))
        chunk = pending.iloc[start:stop]
        print(f"Starting images {start + 1:,}-{stop:,} of {len(pending):,} pending images")
        chunk_results = process_dataset(
            chunk,
            chunk_csv,
            save_comparisons=False,
            report=False,
        )
        chunk_sources = set(chunk["image"])
        if not chunk_sources.issubset(set(chunk_results["source_image"])):
            missing = chunk_sources - set(chunk_results["source_image"])
            raise RuntimeError(f"Chunk ended without result rows for {len(missing)} images.")
        accumulated = pd.concat([accumulated, chunk_results], ignore_index=True)
        temporary = save_csv.with_suffix(save_csv.suffix + ".tmp")
        accumulated.to_csv(temporary, index=False)
        # Atomic replacement prevents a partially written resume file.
        temporary.replace(save_csv)
        chunk_csv.unlink(missing_ok=True)
        print(
            f"Checkpoint saved: {stop:,}/{len(pending):,} pending images; "
            f"{len(accumulated):,} total result rows"
        )
        del chunk_results
        gc.collect()
        if CUDA:
            torch.cuda.empty_cache()

    return accumulated


## 5. Run a reproducible visual-review sample

Sampling is independent for every `(product class, condition)` group, using
seed 42. This gives broad coverage before committing to full-dataset inference.

Review the generated comparison panels for:

- Correct product-class detections.
- Masks that follow object boundaries.
- Reasonable handling of multi-object images.
- Correct overlap between predicted objects and anomaly masks.

Set `REUSE_SAMPLE_RESULTS=False` when the model, thresholds, or input data have
changed.


In [5]:
# Select the review sample and run or reuse its inference results
rng = random.Random(SEED)
# Store source DataFrame indices so sampled images can be excluded later.
sample_indices = []
for (class_name, condition), group in FULL_MANIFEST.groupby(
    ["class", "condition"], sort=True
):
    sample_size = min(SAMPLE_PER_CONDITION, len(group))
    if sample_size < SAMPLE_PER_CONDITION:
        warnings.warn(
            f"{class_name}/{condition} has only {len(group)} images; sampling all of them."
        )
    sample_indices.extend(rng.sample(group.index.tolist(), sample_size))

SAMPLE_MANIFEST = (
    FULL_MANIFEST.loc[sample_indices]
    .sort_values(["class", "condition", "image"])
    .copy()
)
# Prevent duplicate inference by removing sample images from the full pass.
REMAINING_MANIFEST = FULL_MANIFEST.drop(index=sample_indices).copy()
display(
    SAMPLE_MANIFEST.groupby(["class", "condition"]).size().unstack(fill_value=0)
)
print(
    f"Sample images: {len(SAMPLE_MANIFEST):,}; "
    f"remaining images: {len(REMAINING_MANIFEST):,}"
)
if REUSE_SAMPLE_RESULTS and RESULTS_SAMPLE.is_file():
    sample_results = pd.read_csv(RESULTS_SAMPLE)
    print(f"Reusing {len(sample_results):,} existing sample rows from {RESULTS_SAMPLE}")
else:
    sample_results = process_dataset(
        SAMPLE_MANIFEST, RESULTS_SAMPLE, save_comparisons=True
    )


condition,Anomaly,Normal
class,,
candle,20,20
capsules,20,20
cashew,20,20
chewinggum,20,20
fryum,20,20
macaroni1,20,20
macaroni2,20,20
pcb1,20,20
pcb2,20,20


Sample images: 480; remaining images: 10,341
Reusing 1,606 existing sample rows from /home/student/Deep-Mtech/misc/ridac/object_defect_dataset/sample_results.csv


## 6. Process all remaining images and combine metadata

Start this stage only after the sample panels look correct. The resumable
processor skips source images already present in `remaining_results.csv`.

At completion, inspect both displayed summaries:

- `status` counts reveal unreadable images or missed expected classes.
- `assigned_label` counts reveal the normal/defective imbalance that notebook
  03 addresses.


In [6]:
# Run checkpointed full-dataset inference and write full_results.csv
remaining_results = process_dataset_resumable(
    REMAINING_MANIFEST,
    RESULTS_REMAINING,
    chunk_size=PREDICTION_CHUNK_SIZE,
)
sample_results = pd.read_csv(RESULTS_SAMPLE)
# Concatenate disjoint sample and remaining sets into one canonical table.
FULL_RESULTS = pd.concat([sample_results, remaining_results], ignore_index=True)
FULL_RESULTS.to_csv(RESULTS_FULL, index=False)
print(f"Combined results saved to {RESULTS_FULL}")
display(FULL_RESULTS["status"].value_counts(dropna=False).rename("rows"))
display(
    FULL_RESULTS.loc[FULL_RESULTS["status"] == "ok", "assigned_label"]
    .value_counts()
    .rename("saved crops")
)


Resume state: 0 images completed; 10,341 images remaining.


Checkpointed chunks:   0%|          | 0/162 [00:00<?, ?chunk/s]

Starting images 1-64 of 10,341 pending images
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.


Segmenting:   0%|          | 0/64 [00:00<?, ?image/s]

Saved 256 result rows to /home/student/Deep-Mtech/misc/ridac/object_defect_dataset/remaining_results.current_chunk.csv
Checkpoint saved: 64/10,341 pending images; 256 total result rows
Starting images 65-128 of 10,341 pending images
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.


...

Saved 64 result rows to /home/student/Deep-Mtech/misc/ridac/object_defect_dataset/remaining_results.current_chunk.csv
Checkpoint saved: 10,304/10,341 pending images; 32,466 total result rows
Starting images 10,305-10,341 of 10,341 pending images
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.


Segmenting:   0%|          | 0/37 [00:00<?, ?image/s]

Saved 39 result rows to /home/student/Deep-Mtech/misc/ridac/object_defect_dataset/remaining_results.current_chunk.csv
Checkpoint saved: 10,341/10,341 pending images; 32,505 total result rows
Combined results saved to /home/student/Deep-Mtech/misc/ridac/object_defect_dataset/full_results.csv


status
ok    34111
Name: rows, dtype: int64

assigned_label
normal       32890
defective     1221
Name: saved crops, dtype: int64

## 7. Handoff to classifier preparation

The next notebook,
`03_balance_classifier_dataset.ipynb`, reads `full_results.csv` and the saved
crop paths. Preserve the metadata CSVs because they provide source-image groups
needed to prevent train/test leakage.
